# Lesson 06: Deep learning with PyTorch

*Companion notebook for [the lesson page](../lessons/06-pytorch.html). Worked solutions are in `solutions/06-pytorch-solutions.ipynb`; try each exercise first.*

Move from hand-written NumPy to PyTorch, the framework behind most modern AI research and open models; learn tensors, autograd and the standard training loop, then train CNNs and fine-tune a pretrained network.

**Runs on:** CPU works; a GPU runtime (for example Google Colab) makes the CNN sections much faster.

**How to use it:** read a section of the lesson, then run the matching cells here, in order. Change numbers, break things on purpose, and note what happens in your learning log. Cells that need an API key, a GPU or a download say so in their first line.

## Setup

Install what this lesson needs. Skip this cell if your environment from lesson 00 already has these packages.

In [ ]:
%pip install -q torch torchvision scikit-learn

In [ ]:
import numpy as np

## Setup

Install PyTorch from the selector on [pytorch.org](https://pytorch.org/get-started/locally/), which gives the right command for your OS and GPU. On Google Colab it is already installed; switch the runtime to a GPU for the CNN sessions.

Run in a terminal:

```bash
uv add torch torchvision          # or: pip install torch torchvision
```

In [ ]:
# Needs PyTorch; a GPU runtime is much faster
import torch
print(torch.__version__, torch.cuda.is_available())
device = ("cuda" if torch.cuda.is_available()
          else "mps" if torch.backends.mps.is_available()     # Apple silicon
          else "cpu")
print(device)

## Tensors and autograd

### Tensors are NumPy arrays that can live on a GPU and remember gradients

In [ ]:
# Needs PyTorch; a GPU runtime is much faster
x = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
y = torch.randn(2, 3)                     # normal random, shape (2, 3)
z = x @ y                                 # (2, 2) @ (2, 3) -> (2, 3), same rules as NumPy
print(z.shape, z.dtype)                   # torch.Size([2, 3]) torch.float32

a = torch.from_numpy(np.arange(6.0))      # shares memory with the NumPy array
b = z.detach().cpu().numpy()              # back to NumPy (must be on CPU, without grad tracking)

x_gpu = x.to(device)                      # move to GPU; all operands must be on the same device

### Autograd: backprop for free

Mark a tensor with `requires_grad=True` and PyTorch records every operation on it. Calling `.backward()` on a scalar result walks that record backwards and fills in `.grad` for every leaf tensor, which is exactly what you did by hand in lesson 05.

In [ ]:
# Needs PyTorch; a GPU runtime is much faster
w = torch.tensor(2.0, requires_grad=True)
b = torch.tensor(0.5, requires_grad=True)
x = torch.tensor(3.0)
loss = (w * x + b - 10) ** 2              # forward: (6.5 - 10)^2 = 12.25
loss.backward()                           # backward
print(w.grad, b.grad)                     # dL/dw = 2(wx+b-10)x = -21.0, dL/db = -7.0

## Modules, data and the training loop

### nn.Module

Layers and models are subclasses of `nn.Module`. You declare layers in `__init__` and describe the computation in `forward`. PyTorch tracks all their parameters for you.

In [ ]:
# Needs PyTorch; a GPU runtime is much faster
import torch.nn as nn
import torch.nn.functional as F

class MLP(nn.Module):
    def __init__(self, n_in=2, hidden=64, n_out=1, p_drop=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_in, hidden), nn.ReLU(), nn.Dropout(p_drop),
            nn.Linear(hidden, hidden), nn.ReLU(), nn.Dropout(p_drop),
            nn.Linear(hidden, n_out),             # raw logits: no sigmoid here
        )

    def forward(self, x):
        return self.net(x)

model = MLP().to(device)
print(sum(p.numel() for p in model.parameters()))  # 4,417 parameters

> **Output logits, let the loss apply the squashing**
>
> Use `nn.BCEWithLogitsLoss` for binary problems and `nn.CrossEntropyLoss` for multi-class, and feed them **raw logits**. They combine the sigmoid or softmax with the log in a numerically stable way. Applying softmax yourself and then `CrossEntropyLoss` is a classic silent bug: it still trains, just worse.

### Dataset and DataLoader

A `Dataset` knows how to return one example; a `DataLoader` batches, shuffles and loads in parallel.

In [ ]:
# Needs PyTorch; a GPU runtime is much faster
from torch.utils.data import Dataset, DataLoader, TensorDataset
from sklearn.datasets import make_moons

X, y = make_moons(2000, noise=0.2, random_state=0)
X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.float32).unsqueeze(1)        # (2000, 1)
train_ds = TensorDataset(X[:1600], y[:1600])
val_ds = TensorDataset(X[1600:], y[1600:])
train_dl = DataLoader(train_ds, batch_size=64, shuffle=True)
val_dl = DataLoader(val_ds, batch_size=256)

# a custom Dataset only needs __len__ and __getitem__
class SensorWindows(Dataset):
    def __init__(self, series, window=24):
        self.s, self.w = torch.tensor(series, dtype=torch.float32), window
    def __len__(self):
        return len(self.s) - self.w
    def __getitem__(self, i):
        return self.s[i : i + self.w], self.s[i + self.w]      # past window -> next value

### The canonical training loop

Memorise this shape. It is the same for a two-layer MLP and for a billion-parameter transformer.

In [ ]:
# Needs PyTorch; a GPU runtime is much faster
def train(model, train_dl, val_dl, epochs=30, lr=1e-3):
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-2)
    loss_fn = nn.BCEWithLogitsLoss()
    for epoch in range(epochs):
        model.train()                                   # enables dropout etc.
        total = 0.0
        for xb, yb in train_dl:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)                          # 1. forward
            loss = loss_fn(logits, yb)                  # 2. loss
            opt.zero_grad()                             # 3. clear old gradients
            loss.backward()                             # 4. backward
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)   # optional safety
            opt.step()                                  # 5. update
            total += loss.item() * len(xb)

        model.eval()                                    # disables dropout
        correct = 0
        with torch.no_grad():
            for xb, yb in val_dl:
                xb, yb = xb.to(device), yb.to(device)
                correct += ((model(xb) > 0).float() == yb).sum().item()
        if epoch % 5 == 0:
            print(f"epoch {epoch:2d} train loss {total / len(train_dl.dataset):.3f} "
                  f"val acc {correct / len(val_dl.dataset):.3f}")

train(MLP().to(device), train_dl, val_dl)    # val accuracy about 0.97

## Convolutional networks

Even if you mostly work with language, CNN ideas reappear: vision transformers split images into patches, and speech models often start with convolutions over audio.

In [ ]:
# Needs PyTorch; a GPU runtime is much faster
from torchvision import datasets, transforms

tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.2860,), (0.3530,))])
train_set = datasets.FashionMNIST("data", train=True, download=True, transform=tf)
test_set = datasets.FashionMNIST("data", train=False, download=True, transform=tf)
train_dl = DataLoader(train_set, batch_size=128, shuffle=True, num_workers=2)
test_dl = DataLoader(test_set, batch_size=512, num_workers=2)

class SmallCNN(nn.Module):
    def __init__(self, n_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d(2),                                   # 28 -> 14
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2),                                   # 14 -> 7
        )
        self.head = nn.Sequential(nn.Flatten(), nn.Dropout(0.3), nn.Linear(64 * 7 * 7, n_classes))

    def forward(self, x):                                      # x: (B, 1, 28, 28)
        return self.head(self.features(x))                     # (B, 10) logits

model = SmallCNN().to(device)
opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()                                # expects logits and integer labels

for epoch in range(5):
    model.train()
    for xb, yb in train_dl:
        xb, yb = xb.to(device), yb.to(device)
        loss = loss_fn(model(xb), yb)
        opt.zero_grad(); loss.backward(); opt.step()
    model.eval(); correct = 0
    with torch.no_grad():
        for xb, yb in test_dl:
            correct += (model(xb.to(device)).argmax(1).cpu() == yb).sum().item()
    print(epoch, correct / len(test_set))                      # about 0.91 to 0.92 after 5 epochs

## Transfer learning

- **Feature extraction**: freeze the pretrained layers and train only a new final layer. Fast, works with a few hundred images.
- **Fine-tuning**: unfreeze some or all layers and continue training with a small learning rate. Better with more data.

In [ ]:
# Needs PyTorch; a GPU runtime is much faster
from torchvision import models

weights = models.ResNet18_Weights.DEFAULT
backbone = models.resnet18(weights=weights)
for p in backbone.parameters():
    p.requires_grad = False                                 # freeze everything
backbone.fc = nn.Linear(backbone.fc.in_features, 5)          # new head for 5 classes, trainable
backbone = backbone.to(device)
preprocess = weights.transforms()                            # the exact preprocessing it was trained with
opt = torch.optim.AdamW(backbone.fc.parameters(), lr=1e-3)   # only the head's parameters

## Practicalities

### Saving and loading

Save the `state_dict` (a dictionary of parameter tensors), not the whole model object. Include the optimiser state if you want to resume training.

In [ ]:
# Needs PyTorch; a GPU runtime is much faster
torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "epoch": epoch}, "ckpt.pt")

ckpt = torch.load("ckpt.pt", map_location=device)
model = SmallCNN().to(device)
model.load_state_dict(ckpt["model"])
model.eval()

### Mixed precision

Modern GPUs are much faster at 16-bit floating point. Automatic mixed precision runs most operations in bfloat16 or float16 while keeping sensitive ones in float32, often nearly doubling speed and halving memory. LLMs are trained and usually served this way.

In [ ]:
# Needs PyTorch; a GPU runtime is much faster
for xb, yb in train_dl:
    xb, yb = xb.to(device), yb.to(device)
    with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
        loss = loss_fn(model(xb), yb)
    opt.zero_grad(); loss.backward(); opt.step()

### Hands-on: improve the Fashion-MNIST classifier

Starting from `SmallCNN`:

1. Run the "overfit one batch" test and confirm the loss goes near zero.
2. Add data augmentation for training only: `transforms.RandomHorizontalFlip()` and `transforms.RandomCrop(28, padding=2)`.
3. Train for 15 epochs with a cosine learning-rate schedule (`torch.optim.lr_scheduler.CosineAnnealingLR`).
4. Print the confusion matrix on the test set. Which classes does the model confuse, and does that make sense?

In [ ]:
# Your code here

## Check yourself

Take the quiz at the end of [the lesson page](../lessons/06-pytorch.html). Then close everything and write three or four sentences in your learning log about what you learned, from memory.